# 09 — Character-Level Text Generation

Every project so far has been **classification**: given some input, predict which category it belongs to. Positive or negative? Dog or cat? Normal traffic or attack?

Now we flip the script. Instead of labeling things, we're going to **create** things.

**The Goal:** Train a neural network to write text — one character at a time. Feed it a few starting letters, and it continues the sentence.

**Why this matters:** This is the exact same principle behind GPT, Claude, and every modern LLM. They all work by answering one question over and over: *"Given what I've seen so far, what comes next?"* We're building a tiny version of that.

**What you'll learn:**
1. How to turn text into a **sequence prediction** problem (sliding windows)
2. Character-level tokenization — the simplest possible vocabulary
3. Training an LSTM to predict the next character
4. **Temperature sampling** — controlling creativity vs. coherence
5. The crucial difference between **training** (teacher forcing) and **generation** (feeding your own outputs back in)
6. Why modern LLMs are just this idea scaled up 10,000×

**The dataset:** Shakespeare's complete works. By the end, you'll have a model that writes passable Elizabethan English.

## Setup

In [1]:
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'
os.environ['OPENBLAS_NUM_THREADS'] = '1'
os.environ['NUMEXPR_NUM_THREADS'] = '1'

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import numpy as np
import matplotlib.pyplot as plt
import random
from pathlib import Path
import urllib.request
import time

# Device detection
if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')
print(f'Using device: {device}')

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
print(f'Random seed set to {SEED}')

Using device: mps
Random seed set to 42


---
## Part 1: Load and Explore the Text

We'll download Shakespeare's complete works from Project Gutenberg. It's ~5.3 million characters — enough to learn patterns, small enough to train on a laptop.

In [2]:
data_dir = Path.home() / "LocalAI" / "data" / "shakespeare"
data_dir.mkdir(parents=True, exist_ok=True)

file_path = data_dir / "shakespeare.txt"
url = "https://www.gutenberg.org/cache/epub/100/pg100.txt"

if not file_path.exists():
    print("Downloading Shakespeare's complete works (~5.5MB)...")
    urllib.request.urlretrieve(url, file_path)
    print("Done!")
else:
    print("Shakespeare already downloaded.")

# Read the raw text
with open(file_path, 'r', encoding='utf-8') as f:
    raw_text = f.read()

print(f"Total characters: {len(raw_text):,}")
print(f"Unique characters: {len(set(raw_text)):,}")
print()

# The file has Project Gutenberg header/footer boilerplate — let's see what the text looks like
print("=" * 60)
print("FIRST 500 CHARACTERS:")
print("=" * 60)
print(raw_text[:500])

Shakespeare already downloaded.
Total characters: 5,378,702
Unique characters: 108

FIRST 500 CHARACTERS:
﻿The Project Gutenberg eBook of The Complete Works of William Shakespeare
    
This eBook is for the use of anyone anywhere in the United States and
most other parts of the world at no cost and with almost no restrictions
whatsoever. You may copy it, give it away or re-use it under the terms
of the Project Gutenberg License included with this eBook or online
at www.gutenberg.org. If you are not located in the United States,
you will have to check the laws of the country where you are located
bef


In [3]:
# Strip the Gutenberg boilerplate so we only train on actual Shakespeare
start_marker = "*** START OF THE PROJECT GUTENBERG EBOOK"
end_marker = "*** END OF THE PROJECT GUTENBERG EBOOK"

start_idx = raw_text.find(start_marker)
end_idx = raw_text.find(end_marker)

if start_idx != -1 and end_idx != -1:
    # Move past the marker line
    start_idx = raw_text.find('\n', start_idx) + 1
    text = raw_text[start_idx:end_idx].strip()
else:
    # Fallback: just use everything
    text = raw_text

print(f"Cleaned text: {len(text):,} characters")
print()
print("=" * 60)
print("FIRST 800 CHARACTERS OF ACTUAL SHAKESPEARE:")
print("=" * 60)
print(text[:800])
print()
print("...")
print()
print("=" * 60)
print("LAST 500 CHARACTERS:")
print("=" * 60)
print(text[-500:])

Cleaned text: 5,359,342 characters

FIRST 800 CHARACTERS OF ACTUAL SHAKESPEARE:
The Complete Works of William Shakespeare

by William Shakespeare




                    Contents

    THE SONNETS
    ALL’S WELL THAT ENDS WELL
    THE TRAGEDY OF ANTONY AND CLEOPATRA
    AS YOU LIKE IT
    THE COMEDY OF ERRORS
    THE TRAGEDY OF CORIOLANUS
    CYMBELINE
    THE TRAGEDY OF HAMLET, PRINCE OF DENMARK
    THE FIRST PART OF KING HENRY THE FOURTH
    THE SECOND PART OF KING HENRY THE FOURTH
    THE LIFE OF KING HENRY THE FIFTH
    THE FIRST PART OF HENRY THE SIXTH
    THE SECOND PART OF KING HENRY THE SIXTH
    THE THIRD PART OF KING HENRY THE SIXTH
    KING HENRY THE EIGHTH
    THE LIFE AND DEATH OF KING JOHN
    THE TRAGEDY OF JULIUS CAESAR
    THE TRAGEDY OF KING LEAR
    LOVE’S LABOUR’S LOST
    THE TRAGEDY OF MACBETH
    MEASURE FOR MEASURE
    THE MERCHANT OF VENICE
   

...

LAST 500 CHARACTERS:
Lo in this hollow cradle take thy rest,
My throbbing heart shall rock thee day and night:
  

---
## Part 2: Character-Level Tokenization

In Project 08 (IMDB), we tokenized at the **word** level — splitting text into words and building a vocabulary of 25,000 words.

Here we go even simpler: **character-level tokenization.** Every unique character gets its own ID.

| Approach | Vocabulary Size | Sequence Length | Example |
|----------|----------------|-----------------|---------|
| Word-level | 25,000+ | 200 words | "The", "cat", "sat" → [42, 871, 3001] |
| Character-level | ~65-100 | 100 chars | Each unique char gets its own ID (e.g., if vocab = {" ":0, "T":1, "a":2, "c":3, "e":4, "h":5, "s":6, "t":7}, then "The cat sat" → [1, 5, 4, 0, 3, 2, 7, 0, 6, 2, 7]) |

**Why character-level?**
- Tiny vocabulary (~65 unique characters for Shakespeare) — fast to train
- No `<UNK>` token — we see every possible character in the alphabet
- The model must actually learn spelling, punctuation, and capitalization from scratch
- Perfect for learning the core concept before scaling up

In [4]:
# Build character vocabulary
chars = sorted(list(set(text)))
vocab_size = len(chars)

print(f"Vocabulary size: {vocab_size} unique characters")
print()
print("Every character the model knows:")
print(repr(''.join(chars)))
print()

# Create mappings: char <-> ID
char_to_idx = {ch: i for i, ch in enumerate(chars)}
idx_to_char = {i: ch for i, ch in enumerate(chars)}

print("Example mappings:")
for ch in ['a', 'z', 'A', 'Z', ' ', '\n', '.', ',', "'", '!', '?']:
    print(f"  '{ch}' -> ID {char_to_idx.get(ch, 'NOT FOUND')}")

# Convert entire text to integer IDs
text_as_ids = np.array([char_to_idx[ch] for ch in text], dtype=np.int64)
print(f"\nEntire text as integers: {len(text_as_ids):,} numbers")
print(f"First 100 IDs: {text_as_ids[:100]}")
print(f"Decoded back: '{''.join(idx_to_char[i] for i in text_as_ids[:100])}'")

Vocabulary size: 100 unique characters

Every character the model knows:
"\t\n !&'()*,-.0123456789:;?ABCDEFGHIJKLMNOPQRSTUVWXYZ[]_abcdefghijklmnopqrstuvwxyzÀÆÇÉàâæçèéêëîœ—‘’“”…"

Example mappings:
  'a' -> ID 54
  'z' -> ID 79
  'A' -> ID 25
  'Z' -> ID 50
  ' ' -> ID 2
  '
' -> ID 1
  '.' -> ID 11
  ',' -> ID 9
  ''' -> ID 5
  '!' -> ID 3
  '?' -> ID 24

Entire text as integers: 5,359,342 numbers
First 100 IDs: [44 61 58  2 27 68 66 69 65 58 73 58  2 47 68 71 64 72  2 68 59  2 47 62
 65 65 62 54 66  2 43 61 54 64 58 72 69 58 54 71 58  1  1 55 78  2 47 62
 65 65 62 54 66  2 43 61 54 64 58 72 69 58 54 71 58  1  1  1  1  1  2  2
  2  2  2  2  2  2  2  2  2  2  2  2  2  2  2  2  2  2 27 68 67 73 58 67
 73 72  1  1]
Decoded back: 'The Complete Works of William Shakespeare

by William Shakespeare




                    Contents

'


---
## Part 3: The Core Idea — Predicting the Next Character

**Here's the fundamental principle behind ALL modern language models:**

Given a sequence of characters, predict what comes next.

```
Input:  "To be, or not to "
Output: "b"  (because the next word is "be")

Input:  "To be, or not to b"
Output: "e"

Input:  "To be, or not to be"
Output: ","
```

We create training pairs by sliding a window across the text:

```
Text:  "hello world"
       
Window 1: input="hello", target=" "  (space comes after 'hello')
Window 2: input="ello ", target="w"
Window 3: input="llo w", target="o"
Window 4: input="lo wo", target="r"
...and so on
```

**Key insight:** The input and target come from the SAME text, just shifted by one position.
The model's only job is: *look at these N characters, guess the next one.*

In [5]:
# Let's visualize the sliding window
SEQUENCE_LENGTH = 100  # How many characters the model looks at to predict the next one

sample_text = text[1000:1200]  # Grab a chunk
sample_ids = [char_to_idx[ch] for ch in sample_text]

print("Full sample text:")
print(repr(sample_text))
print()

print("Sliding window examples (first 5):")
for i in range(5):
    input_chars = sample_text[i:i+SEQUENCE_LENGTH]
    target_char = sample_text[i+SEQUENCE_LENGTH]
    print(f"  Window {i+1}:")
    print(f"    Input:  {repr(input_chars[:50])}... ({len(input_chars)} chars)")
    print(f"    Target: {repr(target_char)}")
    print()

Full sample text:
'CHARD THE THIRD\n    THE TRAGEDY OF ROMEO AND JULIET\n    THE TAMING OF THE SHREW\n    THE TEMPEST\n    THE LIFE OF TIMON OF ATHENS\n    THE TRAGEDY OF TITUS ANDRONICUS\n    TROILUS AND CRESSIDA\n    TWELFTH'

Sliding window examples (first 5):
  Window 1:
    Input:  'CHARD THE THIRD\n    THE TRAGEDY OF ROMEO AND JULIE'... (100 chars)
    Target: 'T'

  Window 2:
    Input:  'HARD THE THIRD\n    THE TRAGEDY OF ROMEO AND JULIET'... (100 chars)
    Target: 'H'

  Window 3:
    Input:  'ARD THE THIRD\n    THE TRAGEDY OF ROMEO AND JULIET\n'... (100 chars)
    Target: 'E'

  Window 4:
    Input:  'RD THE THIRD\n    THE TRAGEDY OF ROMEO AND JULIET\n '... (100 chars)
    Target: ' '

  Window 5:
    Input:  'D THE THIRD\n    THE TRAGEDY OF ROMEO AND JULIET\n  '... (100 chars)
    Target: 'L'



### Create the PyTorch Dataset

Same pattern as every project, but now the labels are the *next character* in the sequence.

In [6]:
class CharDataset(Dataset):
    """Dataset that creates (input_sequence, next_character) pairs."""
    def __init__(self, text_ids, seq_length):
        self.text_ids = text_ids
        self.seq_length = seq_length
    
    def __len__(self):
        # We can create this many non-overlapping windows
        # (using stride = seq_length for efficiency, or we could use stride=1 for many more)
        return (len(self.text_ids) - self.seq_length) // self.seq_length
    
    def __getitem__(self, idx):
        # Use stride = 1 so every possible window is a training example
        # Start position is random within a chunk for better coverage
        start = idx * self.seq_length
        
        # Input: characters at positions [start : start+seq_length]
        input_seq = self.text_ids[start : start + self.seq_length]
        # Target: characters at positions [start+1 : start+seq_length+1]
        target_seq = self.text_ids[start + 1 : start + self.seq_length + 1]
        
        return (
            torch.tensor(input_seq, dtype=torch.long),
            torch.tensor(target_seq, dtype=torch.long)
        )

# Split into train/validation (90/10)
split_idx = int(len(text_as_ids) * 0.9)
train_ids = text_as_ids[:split_idx]
val_ids = text_as_ids[split_idx:]

print(f"Train characters: {len(train_ids):,}")
print(f"Val characters:   {len(val_ids):,}")

# Create datasets
train_dataset = CharDataset(train_ids, SEQUENCE_LENGTH)
val_dataset = CharDataset(val_ids, SEQUENCE_LENGTH)

print(f"\nTraining examples:   {len(train_dataset):,}")
print(f"Validation examples: {len(val_dataset):,}")

# Look at one example
sample_input, sample_target = train_dataset[0]
print(f"\nExample batch item:")
print(f"  Input shape:  {sample_input.shape}")
print(f"  Target shape: {sample_target.shape}")
print(f"  Input:  {repr(''.join(idx_to_char[i.item()] for i in sample_input[:40]))}...")
print(f"  Target: {repr(''.join(idx_to_char[i.item()] for i in sample_target[:40]))}...")
print()
print("Notice: Target is just the input shifted left by one character!")

Train characters: 4,823,407
Val characters:   535,935

Training examples:   48,233
Validation examples: 5,358

Example batch item:
  Input shape:  torch.Size([100])
  Target shape: torch.Size([100])
  Input:  'The Complete Works of William Shakespear'...
  Target: 'he Complete Works of William Shakespeare'...

Notice: Target is just the input shifted left by one character!


In [7]:
# Create DataLoaders
BATCH_SIZE = 128

train_loader = DataLoader(
    train_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=True,
    drop_last=True  # Drop incomplete last batch
)
val_loader = DataLoader(
    val_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=False
)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches:   {len(val_loader)}")

# Check a batch
batch_input, batch_target = next(iter(train_loader))
print(f"\nBatch input shape:  {batch_input.shape}  (batch_size={BATCH_SIZE}, seq_len={SEQUENCE_LENGTH})")
print(f"Batch target shape: {batch_target.shape}")

Train batches: 376
Val batches:   42

Batch input shape:  torch.Size([128, 100])  (batch_size=128, seq_len=100)
Batch target shape: torch.Size([128, 100])


---
## Part 4: The Model — LSTM for Character Prediction

This is very similar to the LSTM you built in Project 08, but with one critical difference:

**Project 08 (Classification):** LSTM reads the whole review → outputs ONE number (positive/negative)

**Project 09 (Generation):** LSTM reads a sequence → outputs a prediction for EACH position (what character comes after each position)

```
Classification:  [w1, w2, w3, ..., w200] → [0.87]  (one output)
Generation:      [c1, c2, c3, ..., c100] → [ĉ2, ĉ3, ĉ4, ..., ĉ101]  (100 outputs, one per position)
```

This is why we use `lstm_out` (output at every position) instead of just `hidden[-1]` (final hidden state).

In [8]:
class CharLSTM(nn.Module):
    """LSTM-based character-level language model."""
    def __init__(self, vocab_size, embedding_dim=128, hidden_dim=256, n_layers=2, dropout=0.3):
        super().__init__()
        
        # Embedding: each character -> a learnable 128-dim vector
        # The model learns that 'a' and 'e' are similar (both vowels),
        # that ' ' (space) is special, etc.
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        
        # Stacked LSTM: multiple layers for deeper understanding
        # n_layers=2 means the second LSTM reads the first LSTM's output
        self.lstm = nn.LSTM(
            embedding_dim, hidden_dim,
            num_layers=n_layers,
            batch_first=True,
            dropout=dropout  # Dropout between LSTM layers (not on output)
        )
        
        # Dropout for regularization
        self.dropout = nn.Dropout(dropout)
        
        # Final layer: hidden_dim -> vocab_size
        # For each position, predict a score for every possible character
        self.fc = nn.Linear(hidden_dim, vocab_size)
    
    def forward(self, x, hidden=None):
        # x shape: (batch_size, seq_len) — integer character IDs
        batch_size, seq_len = x.shape
        
        # Step 1: Embed characters
        # Shape: (batch_size, seq_len, embedding_dim)
        embedded = self.embedding(x)
        
        # Step 2: Pass through LSTM
        # lstm_out shape: (batch_size, seq_len, hidden_dim)
        # This gives us a hidden state for EVERY position in the sequence
        lstm_out, hidden = self.lstm(embedded, hidden)
        
        # Step 3: Apply dropout
        lstm_out = self.dropout(lstm_out)
        
        # Step 4: Predict next character at each position
        # Shape: (batch_size, seq_len, vocab_size)
        logits = self.fc(lstm_out)
        
        return logits, hidden
    
    def init_hidden(self, batch_size):
        """Initialize LSTM hidden state to zeros."""
        n_layers = self.lstm.num_layers
        hidden_dim = self.lstm.hidden_size
        return (
            torch.zeros(n_layers, batch_size, hidden_dim, device=device),
            torch.zeros(n_layers, batch_size, hidden_dim, device=device)
        )

# Instantiate the model
model = CharLSTM(vocab_size).to(device)
print(model)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTotal parameters:      {total_params:,}")
print(f"Trainable parameters:  {trainable_params:,}")
print(f"\nFor comparison: GPT-3 has 175,000,000,000 parameters")
print(f"Our model has  {total_params / 175e9 * 100:.6f}% of that 😄")

CharLSTM(
  (embedding): Embedding(100, 128)
  (lstm): LSTM(128, 256, num_layers=2, batch_first=True, dropout=0.3)
  (dropout): Dropout(p=0.3, inplace=False)
  (fc): Linear(in_features=256, out_features=100, bias=True)
)

Total parameters:      960,100
Trainable parameters:  960,100

For comparison: GPT-3 has 175,000,000,000 parameters
Our model has  0.000549% of that 😄


---
## Part 5: Training

This is where it gets different from classification. A few key points:

**Loss function:** `CrossEntropyLoss` — but now it's applied to EVERY position in the sequence, not just one final output. The model makes 100 predictions per example (one for each character position), and we average the loss across all of them.

**Teacher forcing:** During training, the model always sees the *correct* previous characters, even if it would have made a mistake. This is crucial — if we let the model feed its own errors back during training, it would never learn properly.

**Evaluation metric:** Perplexity — lower is better. It measures how "surprised" the model is by the actual text. Perplexity of 1 = perfect prediction. Perplexity equal to vocab_size = random guessing.

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Learning rate scheduler: reduce LR when progress stalls
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=2, verbose=True
)

N_EPOCHS = 20
train_losses = []
val_losses = []
val_perplexities = []

print(f"Training on {len(train_loader)} batches per epoch...")
print(f"Model will see each character ~{N_EPOCHS} times\n")

start_time = time.time()

for epoch in range(N_EPOCHS):
    # --- TRAINING ---
    model.train()
    epoch_loss = 0
    
    for batch_input, batch_target in train_loader:
        batch_input = batch_input.to(device)    # (batch, seq_len)
        batch_target = batch_target.to(device)  # (batch, seq_len)
        
        optimizer.zero_grad()
        
        # Forward pass
        # logits: (batch, seq_len, vocab_size)
        logits, _ = model(batch_input)
        
        # Reshape for CrossEntropyLoss:
        # It expects (N, C) for predictions and (N,) for targets
        # where N = batch_size * seq_len (all positions flattened)
        loss = criterion(
            logits.reshape(-1, vocab_size),  # (batch*seq_len, vocab_size)
            batch_target.reshape(-1)          # (batch*seq_len,)
        )
        
        loss.backward()
        
        # Gradient clipping: prevent exploding gradients in RNNs
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        
        optimizer.step()
        epoch_loss += loss.item()
    
    avg_train_loss = epoch_loss / len(train_loader)
    train_losses.append(avg_train_loss)
    
    # --- VALIDATION ---
    model.eval()
    val_loss = 0
    
    with torch.no_grad():
        for batch_input, batch_target in val_loader:
            batch_input = batch_input.to(device)
            batch_target = batch_target.to(device)
            
            logits, _ = model(batch_input)
            loss = criterion(
                logits.reshape(-1, vocab_size),
                batch_target.reshape(-1)
            )
            val_loss += loss.item()
    
    avg_val_loss = val_loss / len(val_loader)
    val_losses.append(avg_val_loss)
    
    # Perplexity = exp(cross-entropy loss)
    # Lower is better. Random guessing = vocab_size (~65)
    perplexity = np.exp(avg_val_loss)
    val_perplexities.append(perplexity)
    
    # Update learning rate based on validation loss
    scheduler.step(avg_val_loss)
    
    elapsed = time.time() - start_time
    print(f'Epoch {epoch+1:2d}/{N_EPOCHS} | '
          f'Train Loss: {avg_train_loss:.3f} | '
          f'Val Loss: {avg_val_loss:.3f} | '
          f'Perplexity: {perplexity:.1f} | '
          f'Time: {elapsed:.0f}s')

total_time = time.time() - start_time
print(f"\nTraining complete! Total time: {total_time:.0f}s ({total_time/60:.1f} min)")
print(f"Final validation perplexity: {val_perplexities[-1]:.1f}")

In [ ]:
# Visualize training progress
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss curves
axes[0].plot(train_losses, label='Train Loss', linewidth=2, color='steelblue')
axes[0].plot(val_losses, label='Val Loss', linewidth=2, color='coral')
axes[0].set_title('Loss Curves', fontsize=14)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Cross-Entropy Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Perplexity
axes[1].plot(val_perplexities, linewidth=2, color='seagreen')
axes[1].axhline(y=vocab_size, color='gray', linestyle='--', alpha=0.5, 
                label=f'Random guessing ({vocab_size})')
axes[1].set_title('Validation Perplexity (lower = better)', fontsize=14)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Perplexity')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Initial perplexity: {val_perplexities[0]:.1f}")
print(f"Final perplexity:   {val_perplexities[-1]:.1f}")
print(f"Improvement:        {val_perplexities[0] - val_perplexities[-1]:.1f} points")
print()
if val_perplexities[-1] < 10:
    print("🍾 Perplexity < 10 — the model has learned strong patterns!")
elif val_perplexities[-1] < 20:
    print("👍 Perplexity < 20 — decent, try training for more epochs")
else:
    print("📚 Perplexity > 20 — model could use more training or a larger architecture")

---
## Part 6: Generating Text

**This is the payoff.** We take our trained model and ask it to write.

The generation process is fundamentally different from training:

**Training (teacher forcing):** Model always sees the *correct* previous characters

**Generation (autoregressive):** Model feeds its OWN predictions back as input

```
Step 1: Give model "ROMEO: " → it predicts "W"
Step 2: Give model "ROMEO: W" → it predicts "h"
Step 3: Give model "ROMEO: Wh" → it predicts "e"
Step 4: Give model "ROMEO: Whe" → it predicts "r"
...and so on for hundreds of characters
```

### Temperature: Controlling Creativity

Temperature controls how "random" the model's predictions are:

| Temperature | Effect | Best For |
|-------------|--------|----------|
| 0.1 — 0.5 | Very predictable, repetitive | When you want safe, boring output |
| 0.7 — 1.0 | Balanced creativity | Most generation tasks |
| 1.2 — 2.0 | Wild, creative, sometimes gibberish | Exploring unusual combinations |

**How it works:** Before sampling, we divide the logits by the temperature. High temperature makes all probabilities more equal (more random). Low temperature amplifies the differences (less random).

In [ ]:
def generate_text(model, start_text, char_to_idx, idx_to_char, 
                  length=500, temperature=0.8, device=device):
    """
    Generate text by repeatedly predicting the next character.
    
    Args:
        model: Trained CharLSTM model
        start_text: Seed text to start generation (e.g., "ROMEO: ")
        char_to_idx: Character to ID mapping
        idx_to_char: ID to character mapping
        length: How many characters to generate
        temperature: Controls randomness (lower = more predictable)
        device: torch device
    
    Returns:
        Full generated text (start_text + generated characters)
    """
    model.eval()
    
    # Convert start text to IDs
    input_ids = [char_to_idx.get(ch, 0) for ch in start_text]
    input_tensor = torch.tensor([input_ids], dtype=torch.long).to(device)
    
    # Initialize hidden state
    hidden = model.init_hidden(1)
    
    generated = list(start_text)
    
    with torch.no_grad():
        # First, process the seed text through the model to build up the hidden state
        if len(input_ids) > 1:
            # Pass all but the last character to warm up the hidden state
            _, hidden = model(input_tensor[:, :-1], hidden)
        
        # Start with the last character of the seed
        current_input = input_tensor[:, -1:]  # Shape: (1, 1)
        
        for _ in range(length):
            # Get model prediction for next character
            logits, hidden = model(current_input, hidden)
            # logits shape: (1, 1, vocab_size) -> squeeze to (vocab_size,)
            logits = logits.squeeze()
            
            # Apply temperature
            # temperature=1.0: use raw probabilities
            # temperature<1.0: amplify high-probability choices (less random)
            # temperature>1.0: flatten probabilities (more random)
            logits = logits / temperature
            
            # Convert logits to probabilities
            probs = torch.softmax(logits, dim=0)
            
            # Sample from the probability distribution
            # Using multinomial means we DON'T always pick the most likely character —
            # we sometimes pick less likely ones, which creates variety
            next_char_id = torch.multinomial(probs, num_samples=1).item()
            
            # Add to generated text
            generated.append(idx_to_char[next_char_id])
            
            # Feed this character back as input for the next step
            current_input = torch.tensor([[next_char_id]], dtype=torch.long).to(device)
    
    return ''.join(generated)

print("Generation function ready!")
print()
print("Let's test it with a quick generation (before much training, it'll be gibberish):")
print()

# Quick test generation
test_output = generate_text(model, "ROMEO: ", char_to_idx, idx_to_char, length=200, temperature=0.8)
print(test_output[:300])

In [ ]:
# Now let's generate with different temperatures and compare
seed_texts = [
    "ROMEO: ",
    "JULIET: ",
    "HAMLET: ",
    "To be, or not to be",
]

print("=" * 70)
print("GENERATED SHAKESPEARE")
print("=" * 70)

for seed in seed_texts:
    print(f"\n{'─' * 70}")
    print(f"SEED: {repr(seed)}")
    print(f"{'─' * 70}")
    generated = generate_text(model, seed, char_to_idx, idx_to_char, 
                              length=400, temperature=0.7)
    print(generated)
    print()

In [ ]:
# Temperature comparison — same seed, different temperatures
seed = "ROMEO: "
temperatures = [0.2, 0.5, 0.8, 1.2, 2.0]

print("=" * 70)
print(f"TEMPERATURE COMPARISON — Seed: {repr(seed)}")
print("=" * 70)

fig, axes = plt.subplots(len(temperatures), 1, figsize=(12, 2.5 * len(temperatures)))

for i, temp in enumerate(temperatures):
    generated = generate_text(model, seed, char_to_idx, idx_to_char, 
                              length=200, temperature=temp)
    
    # Count unique characters (crude measure of diversity)
    unique_ratio = len(set(generated)) / len(generated) * 100
    
    print(f"\n--- Temperature = {temp} ---")
    print(generated[:250])
    print(f"(Unique char ratio: {unique_ratio:.1f}%)")

print()
print("What to notice:")
print("  Low temp (0.2):  Very repetitive, stuck in loops")
print("  Med temp (0.7-0.8): Most natural-looking text")
print("  High temp (2.0): More variety but less coherent")

---
## Part 7: Understanding What the Model Learned

### Visualizing Character Embeddings

The embedding layer has learned relationships between characters. Let's see if vowels cluster together, if punctuation clusters, etc.

In [ ]:
from sklearn.decomposition import PCA

# Extract embedding weights
embedding_weights = model.embedding.weight.data.cpu().numpy()

# Reduce to 2D for visualization
pca = PCA(n_components=2)
embeddings_2d = pca.fit_transform(embedding_weights)

# Plot
fig, ax = plt.subplots(figsize=(16, 12))

# Color-code by character type
for i, ch in enumerate(chars):
    x, y = embeddings_2d[i]
    
    if ch.isupper():
        color, marker, size = 'steelblue', 'o', 60
    elif ch.islower():
        color, marker, size = 'seagreen', 'o', 40
    elif ch.isdigit():
        color, marker, size = 'coral', 's', 60
    elif ch in '.,;:!?':
        color, marker, size = 'purple', 'D', 80
    elif ch in "'\"":
        color, marker, size = 'orange', '^', 50
    else:
        color, marker, size = 'gray', '.', 30
    
    ax.scatter(x, y, c=color, marker=marker, s=size, alpha=0.6, edgecolors='none')
    ax.annotate(ch, (x, y), fontsize=7, alpha=0.7,
                xytext=(3, 3), textcoords='offset points')

ax.set_title('Character Embeddings (PCA Projection)', fontsize=16)
ax.set_xlabel('PC1')
ax.set_ylabel('PC2')

# Legend
from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], marker='o', color='w', markerfacecolor='steelblue', markersize=10, label='Uppercase'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='seagreen', markersize=10, label='Lowercase'),
    Line2D([0], [0], marker='s', color='w', markerfacecolor='coral', markersize=10, label='Digits'),
    Line2D([0], [0], marker='D', color='w', markerfacecolor='purple', markersize=10, label='Punctuation'),
    Line2D([0], [0], marker='^', color='w', markerfacecolor='orange', markersize=10, label='Quotes'),
    Line2D([0], [0], marker='.', color='w', markerfacecolor='gray', markersize=10, label='Other'),
]
ax.legend(handles=legend_elements, loc='upper right')
ax.grid(True, alpha=0.2)

plt.tight_layout()
plt.show()

print("Do you see clusters?")
print("- Vowels near other vowels?")
print("- Punctuation grouped together?")
print("- Uppercase letters forming a separate cluster from lowercase?")

---
## What You Built

1. **Character-level tokenization** — the simplest possible vocabulary (~65 tokens)
2. **Sliding window dataset** — turned raw text into (input_sequence, next_character) training pairs
3. **2-layer LSTM language model** — predicts the next character at every position
4. **Temperature-controlled generation** — sampling from the model's probability distribution
5. **Embedding visualization** — seeing what the model learned about character relationships

## Key Concepts

| Concept | Meaning |
|---------|---------|
| Language Model | A model that assigns probabilities to sequences of text |
| Autoregressive | Generating output one token at a time, feeding predictions back as input |
| Teacher Forcing | During training, always giving the model the correct previous tokens |
| Temperature | Controls the randomness of sampling (low = safe, high = creative) |
| Perplexity | exp(loss) — how "surprised" the model is by real text (lower is better) |
| Sliding Window | Creating training examples by shifting a fixed-size window across text |

## How This Connects to GPT

| Our Model | GPT |
|-----------|-----|
| Character-level tokens | Subword tokens (Byte-Pair Encoding) |
| ~65 token vocabulary | ~50,000 token vocabulary |
| 2-layer LSTM | 96-layer Transformer |
| ~1M parameters | 175B parameters (GPT-3) |
| Shakespeare (~5MB) | Internet-scale text (~45TB) |
| Predicts next character | Predicts next subword token |
| **But the core idea is IDENTICAL** | |

---
## Things to Try

1. **Train on different text** — replace Shakespeare with your own text file (song lyrics, recipes, your chat logs)
2. **Increase sequence length** — does 200 characters produce better text than 100?
3. **Add more LSTM layers** — try 3 or 4 layers and compare perplexity
4. **Experiment with embedding dimensions** — 64 vs 256 vs 512
5. **Implement greedy decoding** — always pick the most likely character instead of sampling (what happens?)
6. **Add a GRU version** — GRU is a simpler alternative to LSTM; compare their performance
7. **Train for more epochs** — leave it running overnight and see how good the text gets
8. **Word-level version** — modify the code to predict words instead of characters (vocabulary of ~10K words)

---
## Bonus: Interactive Generation

Write your own seed text and see what the model writes!

In [ ]:
# Try your own seeds!
custom_seeds = [
    "The king spoke: ",
    "My dearest love, ",
    "Upon the battlefield, ",
    "I do solemnly swear ",
    "All the world's a stage, ",
]

print("=" * 70)
print("YOUR CUSTOM GENERATIONS")
print("=" * 70)

for seed in custom_seeds:
    print(f"\n{'─' * 70}")
    generated = generate_text(model, seed, char_to_idx, idx_to_char, 
                              length=500, temperature=0.8)
    print(generated)
    print()

---
## Bonus 2: Save and Reload Your Model

Training takes time. Save your model so you can generate text anytime without retraining.

In [ ]:
model_dir = Path.home() / "LocalAI" / "models" / "text_generation"
model_dir.mkdir(parents=True, exist_ok=True)

# Save everything needed to regenerate
checkpoint = {
    'model_state_dict': model.state_dict(),
    'char_to_idx': char_to_idx,
    'idx_to_char': idx_to_char,
    'vocab_size': vocab_size,
    'chars': chars,
    'model_config': {
        'embedding_dim': 128,
        'hidden_dim': 256,
        'n_layers': 2,
        'dropout': 0.3
    },
    'val_perplexity': val_perplexities[-1]
}

torch.save(checkpoint, model_dir / 'shakespeare_lstm.pt')
print(f"Model saved to {model_dir / 'shakespeare_lstm.pt'}")
print()

# To reload later:
print("To reload your model in the future:")
print("""
checkpoint = torch.load(model_dir / 'shakespeare_lstm.pt', map_location=device)
model = CharLSTM(vocab_size=checkpoint['vocab_size'], **checkpoint['model_config']).to(device)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

# Then call generate_text() with checkpoint['char_to_idx'] and checkpoint['idx_to_char']
""")